In [1]:
!pip install -q ragas datasets accelerate bitsandbytes
!pip install -q transformers sentencepiece
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360.7 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently tak

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
GG_COLAB = "/content/drive/MyDrive"
LABORLAW_DIR = f'{GG_COLAB}/laborlaw/'


In [4]:
import json
import csv
import gc
import os
import time
import sys
import random
from tqdm import tqdm
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas import RunConfig
from ragas.metrics import (
    # AnswerRelevancy,
    Faithfulness,
    # ContextPrecision,
    # ContextRecall,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import HuggingFaceEmbeddings
from transformers import (AutoTokenizer, AutoModelForCausalLM, pipeline,)
import torch
from langchain_huggingface import (HuggingFacePipeline, HuggingFaceEmbeddings,)
PIPELINE_TO_RUN = "RAG"
GTRUTH_FILE   = f'{LABORLAW_DIR}gtruth_qa.json'
RAG_FILE      = f'{LABORLAW_DIR}rag_qa.json'
GRAPHRAG_FILE = f'{LABORLAW_DIR}graphrag_qa.json'
LLMONLY_FILE  = f'{LABORLAW_DIR}qwen_qa.json'
if PIPELINE_TO_RUN:
    CHECKPOINT_FILE = f'{LABORLAW_DIR}ragas/checkpoint_{PIPELINE_TO_RUN}.json'
    PARTIAL_CSV  = f'{LABORLAW_DIR}ragas/partial_{PIPELINE_TO_RUN}.csv'
    SUMMARY_JSON = f'{LABORLAW_DIR}ragas/summary_{PIPELINE_TO_RUN}.json'
    DETAIL_CSV   = f'{LABORLAW_DIR}ragas/detail_{PIPELINE_TO_RUN}.csv'
else:
    CHECKPOINT_FILE = f'{LABORLAW_DIR}ragas/ragas_checkpoint.json'
    PARTIAL_CSV  = f'{LABORLAW_DIR}ragas/partial_results.csv'
    SUMMARY_JSON = f'{LABORLAW_DIR}ragas/results_summary.json'
    DETAIL_CSV   = f'{LABORLAW_DIR}ragas/results_detail.csv'

JUDGE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

run_config = RunConfig(
    max_workers=1,
    timeout=90,
    max_retries=1,
)
faithfulness = Faithfulness()
BATCH_SIZE = 8
METRIC_KEYS = [
    "answer_relevancy",
    "faithfulness",
    "context_precision",
    "context_recall",
]

print("Loading tokenizer")
tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

print("Loading model")
from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL,
    device_map="auto",
    torch_dtype=torch.float16,   # dtype -> torch_dtype
    quantization_config=bnb_config,
)
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
)
hf_llm = HuggingFacePipeline(pipeline=pipe)
llm = LangchainLLMWrapper(hf_llm)
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
metrics = [Faithfulness()]

#  helpers
def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def load_checkpoint():
    if not os.path.exists(CHECKPOINT_FILE):
        return None
    with open(CHECKPOINT_FILE, encoding="utf-8") as f:
        return json.load(f)

def save_checkpoint(pipeline_name, next_index, skipped_ids=None):
    os.makedirs(os.path.dirname(CHECKPOINT_FILE), exist_ok=True)
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump({
            "pipeline": pipeline_name,
            "next_index": next_index,
            "skipped_ids": skipped_ids or [],
        }, f)

def clear_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)

def append_to_partial_csv(df_batch):
    os.makedirs(os.path.dirname(PARTIAL_CSV), exist_ok=True)
    write_header = not os.path.exists(PARTIAL_CSV)
    df_batch.to_csv(PARTIAL_CSV, mode="a", header=write_header,
                    index=False, encoding="utf-8-sig")

def format_eta(seconds):
    minutes = int(seconds // 60)
    secs = int(seconds % 60)
    return f"{minutes}p {secs}s"

# => data
ground_truth_list = read_json(GTRUTH_FILE)
gtruth_map = {item["id"]: item for item in ground_truth_list}

def build_rows(path, pipeline_name):
    rows = []
    for d in read_json(path):
        if d.get("id") not in gtruth_map:
            continue
        ctx = d.get("context_text") or ""
        contexts = [] if pipeline_name == "LLM_only" else ([ctx] if ctx else [])
        rows.append({
            "id":  d["id"],
            "question":  d["question"],
            "answer":  d["answer"],
            "contexts":  contexts,
            "ground_truth":  gtruth_map[d["id"]]["answer"],
            "question_type": d.get("question_type", "unknown"),
        })
    return rows

pipelines = {
    "RAG": build_rows(RAG_FILE, "RAG"),
    "GraphRAG": build_rows(GRAPHRAG_FILE,"GraphRAG"),
    "LLM_only": build_rows(LLMONLY_FILE, "LLM_only"),
}
if PIPELINE_TO_RUN:
    if PIPELINE_TO_RUN not in pipelines:
        print(f"ERROR: Pipeline '{PIPELINE_TO_RUN}' không tồn tại!")
        sys.exit(1)
    pipelines = {PIPELINE_TO_RUN: pipelines[PIPELINE_TO_RUN]}
else:
    print('Chạy tất cả pipeline!')
print("\nSố câu hỏi:")
for name, rows in pipelines.items():
    print(f"{name}: {len(rows)}")

# ----> eval batch ( fallback từng sample)
def eval_batch(batch_rows, pipeline_name, batch_index):
    # Tầng 1: thử cả batch
    try:
        dataset = Dataset.from_list([{
            "question": r["question"], "answer": r["answer"],
            "contexts": r["contexts"], "ground_truth": r["ground_truth"],
        } for r in batch_rows])
        result = evaluate(dataset, metrics=metrics, llm=llm,
                          embeddings=embeddings, run_config=run_config,
                          raise_exceptions=False)
        return result.to_pandas(), []
    except Exception as e:
        print(f"\n Batch {batch_index} fail: {e} → thử từng sample lẻ...")

    # Tầng 2: từng sample lẻ
    single_dfs, skipped = [], []
    run_single = RunConfig(max_workers=1, timeout=180, max_retries=1)
    for r in batch_rows:
        try:
            ds = Dataset.from_list([{
                "question": r["question"], "answer": r["answer"],
                "contexts": r["contexts"], "ground_truth": r["ground_truth"],
            }])
            res = evaluate(ds, metrics=metrics, llm=llm,
                           embeddings=embeddings, run_config=run_single,
                           raise_exceptions=False)
            single_dfs.append(res.to_pandas())
        except Exception as e2:
            print(f"  Sample {r['id']} fail: {e2} → NaN")
            nan_row = {"question": r["question"], "answer": r["answer"],
                       "faithfulness": float("nan")}
            single_dfs.append(pd.DataFrame([nan_row]))
            skipped.append(r["id"])

    df = pd.concat(single_dfs, ignore_index=True) if single_dfs else None
    return df, skipped

#  main loop
checkpoint = load_checkpoint()
all_dfs = {}

for pipeline_name, rows in pipelines.items():
    start_index = 0
    if checkpoint and checkpoint["pipeline"] == pipeline_name:
        start_index = checkpoint["next_index"]

    print(f"\n===========================================")
    print(f"Pipeline: {pipeline_name}")
    print(f"=====================================================")

    pipeline_dfs   = []
    time_per_batch = []

    for start in tqdm(range(start_index, len(rows), BATCH_SIZE), desc=pipeline_name):
        batch_rows = rows[start:start + BATCH_SIZE]
        batch_index = start // BATCH_SIZE
        t0 = time.time()

        df_batch, skipped_ids = eval_batch(batch_rows, pipeline_name, batch_index)

        if df_batch is None:
            save_checkpoint(pipeline_name, start + BATCH_SIZE, skipped_ids)
            continue

        df_batch["question_type"] = [r["question_type"] for r in batch_rows][:len(df_batch)]
        df_batch["id"] = [r["id"]  for r in batch_rows][:len(df_batch)]
        df_batch["pipeline"] = pipeline_name

        pipeline_dfs.append(df_batch)
        append_to_partial_csv(df_batch)

        elapsed = time.time() - t0
        time_per_batch.append(elapsed)
        avg_time  = sum(time_per_batch) / len(time_per_batch)
        remaining_batches = max(len(rows) - (start + BATCH_SIZE), 0) / BATCH_SIZE
        eta_seconds  = avg_time * remaining_batches

        tqdm.write(f"Batch {batch_index} | {elapsed:.1f}s | ETA {format_eta(eta_seconds)}")

        save_checkpoint(pipeline_name, start + BATCH_SIZE, skipped_ids)

        if batch_index % 3 == 0:
            torch.cuda.empty_cache()
            gc.collect()

    if pipeline_dfs:
        all_dfs[pipeline_name] = pd.concat(pipeline_dfs, ignore_index=True)

#  save
clear_checkpoint()

if not all_dfs:
    print("Không có kết quả.")
    sys.exit(1)

summary = {}
for name, df in all_dfs.items():
    summary[name] = {}
    for k in METRIC_KEYS:
        if k in df.columns:
            summary[name][k] = round(float(df[k].mean()), 4)

print("\n========================")
print("SUMMARY")
print("========================")
for name, vals in summary.items():
    print(f"\n{name}")
    for k, v in vals.items():
        print(f"  {k}: {v}")

output = {
    "judge_model":JUDGE_MODEL,
    "embedding_model": EMBED_MODEL,
    "summary": summary,
}
with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

combined_df = pd.concat(list(all_dfs.values()), ignore_index=True)
combined_df.to_csv(DETAIL_CSV, index=False, encoding="utf-8-sig")

print("\nDONE")
print(SUMMARY_JSON)
print(DETAIL_CSV)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_3615/1955224307.py:13: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (


Loading tokenizer


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
ERRO

Batch 12 | 217.6s | ETA 0p 0s


RAG: 100%|██████████| 13/13 [1:50:07<00:00, 508.24s/it]


SUMMARY

RAG
  faithfulness: 0.05

DONE
/content/drive/MyDrive/laborlaw/ragas/summary_RAG.json
/content/drive/MyDrive/laborlaw/ragas/detail_RAG.csv
